# 🏆 Week 4 · Day 24-25: 실전 프로젝트 — House Prices (회귀)

## 🎯 학습 목표
- **회귀 문제**의 전체 파이프라인 체화
- **RMSLE** 평가지표 대응 (`log1p` 변환)
- **고차원 피처** 처리 전략
- 회귀에 특화된 Feature Engineering

> 📌 **대회 링크**: https://www.kaggle.com/c/house-prices-advanced-regression-techniques


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import KFold, train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.linear_model import Ridge, Lasso, ElasticNet
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error
import lightgbm as lgb
import xgboost as xgb
from catboost import CatBoostRegressor
from scipy.stats import skew
import warnings; warnings.filterwarnings('ignore')
np.random.seed(42)

## 1. 데이터 준비

실전: `train = pd.read_csv('../data/house-prices/train.csv')`

여기서는 재현 가능한 합성 데이터로 시연

In [ ]:
# House Prices와 유사한 스키마의 합성 데이터
def make_houses():
    np.random.seed(42)
    n_train, n_test = 1460, 1459
    def make_df(n, target=True):
        df = pd.DataFrame({
            'Id': np.arange(1, n+1),
            'OverallQual':  np.random.randint(1, 11, n),
            'GrLivArea':    np.random.normal(1500, 500, n).clip(400, 5000),
            'GarageCars':   np.random.randint(0, 5, n),
            'TotalBsmtSF':  np.random.normal(1000, 400, n).clip(0, 6000),
            'YearBuilt':    np.random.randint(1900, 2011, n),
            'YearRemod':    np.random.randint(1950, 2011, n),
            '1stFlrSF':     np.random.normal(1100, 350, n).clip(400, 4000),
            '2ndFlrSF':     np.where(np.random.rand(n)<0.6, np.random.normal(600, 300, n).clip(0,2000), 0),
            'FullBath':     np.random.randint(0, 4, n),
            'BedroomAbvGr': np.random.randint(0, 6, n),
            'TotRmsAbvGrd': np.random.randint(2, 14, n),
            'LotArea':      np.random.lognormal(9.0, 0.5, n).clip(1000, 100000),
            'Neighborhood': np.random.choice(
                ['CollgCr','Veenker','Crawfor','NoRidge','Mitchel','Somerst','NWAmes',
                 'OldTown','BrkSide','Sawyer','NridgHt','NAmes','SawyerW','IDOTRR'], n),
            'KitchenQual':  np.random.choice(['Ex','Gd','TA','Fa'], n, p=[0.1,0.4,0.45,0.05]),
            'ExterQual':    np.random.choice(['Ex','Gd','TA','Fa'], n, p=[0.05,0.3,0.6,0.05]),
            'BsmtQual':     np.random.choice(['Ex','Gd','TA','Fa','NA'], n, p=[0.1,0.35,0.4,0.05,0.1]),
            'GarageType':   np.random.choice(['Attchd','Detchd','BuiltIn','NA'], n, p=[0.6,0.25,0.1,0.05]),
            'MSZoning':     np.random.choice(['RL','RM','FV','RH','C'], n, p=[0.77,0.15,0.04,0.02,0.02]),
        })
        # 결측치 주입
        for col, frac in [('GarageType', 0.05), ('BsmtQual', 0.03)]:
            df.loc[np.random.choice(n, int(n*frac), replace=False), col] = np.nan
        if target:
            # SalePrice ~ OverallQual과 GrLivArea가 핵심
            price = (50000
                     + 25000 * df['OverallQual']
                     + 70 * df['GrLivArea']
                     + 50 * df['TotalBsmtSF']
                     + 15000 * df['GarageCars']
                     + 200 * (df['YearBuilt'] - 1900)
                     + np.random.lognormal(0, 0.15, n) * 20000)
            df['SalePrice'] = price.clip(30000, 800000).astype(int)
        return df
    return make_df(n_train), make_df(n_test, False)

train, test = make_houses()
print("Train:", train.shape, " Test:", test.shape)
train.head(3)

## 2. 타깃 변수 분석 — 회귀에서 가장 먼저 봐야 할 것

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(13, 4))
train['SalePrice'].hist(bins=50, ax=ax[0], color='steelblue')
ax[0].set_title(f'SalePrice (skew={skew(train["SalePrice"]):.2f})')

np.log1p(train['SalePrice']).hist(bins=50, ax=ax[1], color='seagreen')
ax[1].set_title(f'log1p(SalePrice) (skew={skew(np.log1p(train["SalePrice"])):.2f})')
plt.tight_layout(); plt.show()

print("→ log 변환 후 거의 정규분포 → RMSLE 지표에 맞춰 log1p로 학습")

## 3. 수치형 피처의 왜도 교정

In [ ]:
num_cols = train.select_dtypes(include='number').columns.drop(['Id','SalePrice'])
skews = train[num_cols].apply(lambda x: skew(x.dropna())).sort_values(ascending=False)
print("|skew| > 0.75 피처:")
high_skew = skews[skews.abs() > 0.75].index.tolist()
print(high_skew)

In [ ]:
# log1p로 변환
for col in high_skew:
    if (train[col] >= 0).all():
        train[col] = np.log1p(train[col])
        test[col]  = np.log1p(test[col])

## 4. Feature Engineering

In [ ]:
def feature_engineering(df):
    df = df.copy()

    # 1. 총 면적
    df['TotalSF'] = df['TotalBsmtSF'] + df['1stFlrSF'] + df['2ndFlrSF']

    # 2. 건물 나이
    df['HouseAge']    = 2011 - df['YearBuilt']
    df['RemodelAge']  = 2011 - df['YearRemod']
    df['IsRemodeled'] = (df['YearBuilt'] != df['YearRemod']).astype(int)

    # 3. 평균 방 크기
    df['AvgRoomSize'] = df['GrLivArea'] / df['TotRmsAbvGrd'].clip(lower=1)

    # 4. 2층 있는지
    df['Has2ndFlr'] = (df['2ndFlrSF'] > 0).astype(int)
    df['HasBsmt']   = (df['TotalBsmtSF'] > 0).astype(int)

    # 5. 품질 점수 합
    qual_map = {'Ex': 5, 'Gd': 4, 'TA': 3, 'Fa': 2, 'Po': 1, 'NA': 0}
    for col in ['KitchenQual','ExterQual','BsmtQual']:
        df[f'{col}_num'] = df[col].fillna('NA').map(qual_map)
    df['QualSum'] = df['KitchenQual_num'] + df['ExterQual_num'] + df['BsmtQual_num']

    return df

train_fe = feature_engineering(train)
test_fe  = feature_engineering(test)
print("After FE shape:", train_fe.shape)

## 5. 결측치 처리 & 인코딩

In [ ]:
# 전체 결측치 확인
for name, df in [('train', train_fe), ('test', test_fe)]:
    m = df.isna().sum()
    if m.sum() > 0:
        print(f"[{name}]\n{m[m>0]}\n")

# 범주형: 'Missing'으로 대체
cat_cols = train_fe.select_dtypes(include='object').columns
for c in cat_cols:
    train_fe[c] = train_fe[c].fillna('Missing')
    test_fe[c]  = test_fe[c].fillna('Missing')

# 수치형: 중앙값
num_cols = train_fe.select_dtypes(include='number').columns.drop(['Id','SalePrice'])
for c in num_cols:
    med = train_fe[c].median()
    train_fe[c] = train_fe[c].fillna(med)
    test_fe[c]  = test_fe[c].fillna(med)

print("결측 처리 완료")

In [ ]:
# 범주형 Label Encoding
for col in cat_cols:
    le = LabelEncoder()
    combined = pd.concat([train_fe[col], test_fe[col]]).astype(str)
    le.fit(combined)
    train_fe[col] = le.transform(train_fe[col].astype(str))
    test_fe[col]  = le.transform(test_fe[col].astype(str))

## 6. 모델 학습 (log 타깃 사용)

In [ ]:
drop_cols = ['Id','SalePrice']
features = [c for c in train_fe.columns if c not in drop_cols]
print(f"사용 feature: {len(features)}개")

X = train_fe[features].values
y = np.log1p(train_fe['SalePrice'].values)   # ← log 변환
X_test = test_fe[features].values

def rmse_score(y_true, y_pred):
    return np.sqrt(mean_squared_error(y_true, y_pred))

def train_kfold_reg(model_class, params, X, y, X_test, n_splits=5, name=''):
    oof = np.zeros(len(y))
    test_pred = np.zeros(len(X_test))
    kf = KFold(n_splits=n_splits, shuffle=True, random_state=42)

    for fold, (tr, val) in enumerate(kf.split(X)):
        m = model_class(**params)
        if 'lgb' in model_class.__module__:
            m.fit(X[tr], y[tr], eval_set=[(X[val], y[val])],
                  callbacks=[lgb.early_stopping(50), lgb.log_evaluation(0)])
        elif 'xgboost' in model_class.__module__:
            m.fit(X[tr], y[tr], eval_set=[(X[val], y[val])], verbose=False)
        elif 'catboost' in model_class.__module__:
            m.fit(X[tr], y[tr], eval_set=(X[val], y[val]), verbose=0)
        else:
            m.fit(X[tr], y[tr])
        oof[val] = m.predict(X[val])
        test_pred += m.predict(X_test) / n_splits

    print(f"{name:15s} OOF RMSE(log): {rmse_score(y, oof):.5f}")
    return oof, test_pred

In [ ]:
# LightGBM
lgb_params = dict(n_estimators=2000, learning_rate=0.03, num_leaves=31,
                  min_child_samples=10, subsample=0.8, colsample_bytree=0.8,
                  reg_alpha=0.1, reg_lambda=0.1,
                  random_state=42, verbose=-1, n_jobs=-1)
oof_lgb, test_lgb = train_kfold_reg(lgb.LGBMRegressor, lgb_params, X, y, X_test, name='LightGBM')

In [ ]:
# XGBoost
xgb_params = dict(n_estimators=2000, learning_rate=0.03, max_depth=5,
                  min_child_weight=1, subsample=0.8, colsample_bytree=0.8,
                  reg_alpha=0.1, reg_lambda=1.0, early_stopping_rounds=50,
                  random_state=42, n_jobs=-1)
oof_xgb, test_xgb = train_kfold_reg(xgb.XGBRegressor, xgb_params, X, y, X_test, name='XGBoost')

In [ ]:
# CatBoost
cb_params = dict(iterations=2000, learning_rate=0.03, depth=6,
                 l2_leaf_reg=3, random_seed=42, early_stopping_rounds=50, verbose=0)
oof_cb, test_cb = train_kfold_reg(CatBoostRegressor, cb_params, X, y, X_test, name='CatBoost')

In [ ]:
# Ridge (선형 모델은 스케일링 필요)
from sklearn.preprocessing import StandardScaler
sc = StandardScaler()
X_sc  = sc.fit_transform(X)
Xt_sc = sc.transform(X_test)
oof_ridge, test_ridge = train_kfold_reg(Ridge, dict(alpha=10.0, random_state=42),
                                         X_sc, y, Xt_sc, name='Ridge')

## 7. 앙상블 (가중 평균 + Stacking)

In [ ]:
# 7.1 Simple weighted avg (CV 점수 기반)
weights = {'lgb': 0.35, 'xgb': 0.25, 'cb': 0.3, 'ridge': 0.1}
oof_avg  = weights['lgb']*oof_lgb + weights['xgb']*oof_xgb + weights['cb']*oof_cb + weights['ridge']*oof_ridge
test_avg = weights['lgb']*test_lgb + weights['xgb']*test_xgb + weights['cb']*test_cb + weights['ridge']*test_ridge
print(f"Weighted Avg OOF RMSE: {rmse_score(y, oof_avg):.5f}")

In [ ]:
# 7.2 Stacking
oof_stack  = np.column_stack([oof_lgb, oof_xgb, oof_cb, oof_ridge])
test_stack = np.column_stack([test_lgb, test_xgb, test_cb, test_ridge])

meta = Ridge(alpha=1.0)
meta.fit(oof_stack, y)
oof_meta  = meta.predict(oof_stack)
test_meta = meta.predict(test_stack)
print(f"Stacking OOF RMSE: {rmse_score(y, oof_meta):.5f}")
print(f"Meta coefs: {dict(zip(['lgb','xgb','cb','ridge'], meta.coef_.round(3)))}")

## 8. 제출 파일 생성

In [ ]:
# log → 원본 스케일 복원
final_pred = np.expm1(test_meta)

submission = pd.DataFrame({'Id': test['Id'], 'SalePrice': final_pred})
submission.to_csv('../data/house_prices_submission.csv', index=False)
print(submission.head())
print(f"\n제출 파일 생성 완료: {submission.shape}")

### 제출 명령

```bash
kaggle competitions submit -c house-prices-advanced-regression-techniques \
       -f ../data/house_prices_submission.csv \
       -m "Stacking: LGBM+XGB+CB+Ridge with FE, log-target"
```

## 9. 개선 포인트
- Target Encoding for Neighborhood
- Polynomial features on top QualSum, TotalSF, GrLivArea
- 이상치 제거 (GrLivArea > 4000 같은 것)
- Elastic Net / LassoLars / Huber loss
- Kernel Ridge Regression
- Isotonic Regression 으로 post-processing

---

## 📝 Day 24-25 체크리스트
- [ ] 회귀 타깃 log 변환
- [ ] 수치형 왜도 처리
- [ ] 4종 모델 앙상블 가능
- [ ] RMSE를 log 스케일에서 계산 → 제출 시 expm1 복원

다음 노트북에서 **딥러닝 입문 (Keras)** 과 **이미지/텍스트 대회 전략**을 다룹니다.